In [ ]:
# ==========================================================
# 04_model_training_baseline_models.ipynb
# Baseline model training with reduced features
# ==========================================================

import pandas as pd
import numpy as np
import os
import json
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


Dataset shape: (1470, 13) Target distribution: {0: 1233, 1: 237}
Train shape: (1176, 13) Test shape: (294, 13)

===== Logistic Regression =====


C:\Users\Prathamesh16\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.7585034013605442
Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.75      0.84       247
           1       0.38      0.81      0.52        47

    accuracy                           0.76       294
   macro avg       0.67      0.78      0.68       294
weighted avg       0.86      0.76      0.79       294

✅ Saved model: ../models/logistic_regression.pkl

===== Decision Tree =====
Accuracy: 0.7551020408163265
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.79      0.84       247
           1       0.34      0.55      0.42        47

    accuracy                           0.76       294
   macro avg       0.62      0.67      0.63       294
weighted avg       0.81      0.76      0.78       294

✅ Saved model: ../models/decision_tree.pkl

===== Random Forest =====
Accuracy: 0.8435374149659864
Classification Report:
               precision    recall  f1-sco

In [ ]:

# ==========================================================
# 🔽 Load reduced dataset
# ==========================================================
df = pd.read_csv("../data/processed/reduced_features.csv")

X = df.drop(columns=["Attrition_encoded"])
y = df["Attrition_encoded"]

print("Dataset shape:", X.shape, "Target distribution:", y.value_counts().to_dict())


In [ ]:

# ==========================================================
# 🔽 Train/test split
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


In [ ]:

# ==========================================================
# 🔽 Define baseline models
# ==========================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

results = {}

# ==========================================================
# 🔽 Train and evaluate models
# ==========================================================
for name, model in models.items():
    print(f"\n===== {name} =====")
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print("Accuracy:", acc)
    print("Classification Report:\n", classification_report(y_test, y_pred))
    
    results[name] = acc
    
    # Save each model
    model_path = f"../models/{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(model, model_path)
    print(f"✅ Saved model: {model_path}")


In [ ]:

# ==========================================================
# 🔽 Save best model
# ==========================================================
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

joblib.dump(best_model, "../models/best_model.pkl")
print(f"\n🎉 Best model: {best_model_name} with accuracy {results[best_model_name]:.4f}")
print("✅ Saved best model to ../models/best_model.pkl")

# ==========================================================
# 🔽 Save feature manifest
# ==========================================================
manifest = {"features": list(X.columns)}
with open("../models/feature_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("✅ Feature manifest saved:", manifest)
